# Đánh giá mô hình dự đoán tuổi Abalone

Sổ tay này đánh giá chi tiết mô hình đã huấn luyện ở bước 03, bao gồm hiệu năng tổng thể, phân tích phần dư, kiểm tra độ ổn định và phân tích sai số theo từng nhóm dữ liệu.

## 1. Mục tiêu đánh giá

- Đọc kết quả huấn luyện từ notebook 03 (nếu có).
- Tái tạo mô hình tốt nhất khi chưa có tệp kết quả đã lưu.
- Đánh giá trên tập kiểm tra với các chỉ số `MAE`, `RMSE`, `R2`.
- Phân tích phần dư, độ ổn định theo bootstrap và sai số theo nhóm `Sex`.

## 2. Import thư viện và cấu hình

In [ ]:
import json
import pickle
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", 100)

## 3. Thiết lập đường dẫn và nạp dữ liệu đánh giá

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
INTERIM_DATA_PATH = PROJECT_ROOT / "data" / "interim"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"
OUTPUT_MODELS_PATH = PROJECT_ROOT / "outputs" / "models"
OUTPUT_METRICS_PATH = PROJECT_ROOT / "outputs" / "metrics"
OUTPUT_FIGURES_PATH = PROJECT_ROOT / "outputs" / "figures"

OUTPUT_MODELS_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_METRICS_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES_PATH.mkdir(parents=True, exist_ok=True)

MODEL_PATH = OUTPUT_MODELS_PATH / "best_regression_model.pkl"
RESULTS_PATH = OUTPUT_METRICS_PATH / "model_comparison_results.csv"
SUMMARY_PATH = OUTPUT_METRICS_PATH / "best_model_summary.json"

print(f"Đã có tệp mô hình: {MODEL_PATH.exists()}")
print(f"Đã có bảng kết quả: {RESULTS_PATH.exists()}")
print(f"Đã có tệp tóm tắt: {SUMMARY_PATH.exists()}")

In [ ]:
def load_pickle(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)

baseline_scaled = load_pickle(INTERIM_DATA_PATH / "baseline_scaled.pkl")
iqr_scaled = load_pickle(INTERIM_DATA_PATH / "iqr_filtered_scaled.pkl")
log_scaled = load_pickle(INTERIM_DATA_PATH / "log_transformed_scaled.pkl")
fe_scaled = load_pickle(PROCESSED_DATA_PATH / "baseline_with_features_scaled.pkl")

datasets = {
    "baseline_standard": {
        "X_train": baseline_scaled["standard"]["X_train"],
        "X_test": baseline_scaled["standard"]["X_test"],
        "y_train": baseline_scaled["y_train"],
        "y_test": baseline_scaled["y_test"],
    },
    "baseline_robust": {
        "X_train": baseline_scaled["robust"]["X_train"],
        "X_test": baseline_scaled["robust"]["X_test"],
        "y_train": baseline_scaled["y_train"],
        "y_test": baseline_scaled["y_test"],
    },
    "iqr_standard": {
        "X_train": iqr_scaled["standard"]["X_train"],
        "X_test": iqr_scaled["standard"]["X_test"],
        "y_train": iqr_scaled["y_train"],
        "y_test": iqr_scaled["y_test"],
    },
    "log_standard": {
        "X_train": log_scaled["standard"]["X_train"],
        "X_test": log_scaled["standard"]["X_test"],
        "y_train": log_scaled["y_train"],
        "y_test": log_scaled["y_test"],
    },
    "feature_engineering_standard": {
        "X_train": fe_scaled["X_train"],
        "X_test": fe_scaled["X_test"],
        "y_train": fe_scaled["y_train"],
        "y_test": fe_scaled["y_test"],
    },
}

for name, item in datasets.items():
    print(f"{name:30s} | train={item['X_train'].shape} | test={item['X_test'].shape}")

## 4. Chọn mô hình để đánh giá

Ưu tiên dùng mô hình đã lưu từ notebook 03. Nếu chưa có tệp kết quả, sổ tay sẽ tự huấn luyện nhanh để tìm cấu hình tốt nhất theo `RMSE`.

In [ ]:
def make_model(model_name: str, random_state: int = 42):
    if model_name == "LinearRegression":
        return LinearRegression()
    if model_name == "Ridge":
        return Ridge(alpha=1.0, random_state=random_state)
    if model_name == "Lasso":
        return Lasso(alpha=0.001, random_state=random_state)
    if model_name == "RandomForest":
        return RandomForestRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_split=4,
            min_samples_leaf=2,
            random_state=random_state,
            n_jobs=-1,
        )
    if model_name == "GradientBoosting":
        return GradientBoostingRegressor(random_state=random_state)
    raise ValueError(f"Unsupported model: {model_name}")


def compute_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_true, y_pred),
    }

In [ ]:
if MODEL_PATH.exists() and SUMMARY_PATH.exists():
    with open(MODEL_PATH, "rb") as f:
        best_model = pickle.load(f)

    with open(SUMMARY_PATH, "r", encoding="utf-8") as f:
        best_summary = json.load(f)

    best_dataset_name = best_summary["best_dataset"]
    best_model_name = best_summary["best_model"]
    selected_data = datasets[best_dataset_name]
    print("Đã nạp mô hình từ thư mục outputs/.")
else:
    print("Chưa có tệp kết quả từ notebook 03 -> bắt đầu huấn luyện nhanh để chọn mô hình tốt nhất.")

    candidate_models = ["LinearRegression", "Ridge", "Lasso", "RandomForest", "GradientBoosting"]
    rows = []

    for dataset_name, data in datasets.items():
        for model_name in candidate_models:
            model = make_model(model_name)
            model.fit(data["X_train"], data["y_train"])
            pred = model.predict(data["X_test"])
            metrics = compute_metrics(data["y_test"], pred)
            rows.append(
                {
                    "dataset": dataset_name,
                    "model": model_name,
                    "test_mae": metrics["MAE"],
                    "test_rmse": metrics["RMSE"],
                    "test_r2": metrics["R2"],
                }
            )

    tmp_df = pd.DataFrame(rows).sort_values(["test_rmse", "test_mae"], ascending=True).reset_index(drop=True)
    best_row = tmp_df.iloc[0]
    best_dataset_name = best_row["dataset"]
    best_model_name = best_row["model"]

    selected_data = datasets[best_dataset_name]
    best_model = make_model(best_model_name)
    best_model.fit(selected_data["X_train"], selected_data["y_train"])

    best_summary = {
        "best_dataset": best_dataset_name,
        "best_model": best_model_name,
        "test_mae": float(best_row["test_mae"]),
        "test_rmse": float(best_row["test_rmse"]),
        "test_r2": float(best_row["test_r2"]),
        "timestamp": datetime.now().strftime("%Y%m%d_%H%M%S"),
    }

print(f"Bộ dữ liệu đánh giá: {best_dataset_name}")
print(f"Mô hình đánh giá: {best_model_name}")

## 5. Đánh giá hiệu năng tổng thể trên tập test

In [ ]:
X_test = selected_data["X_test"]
y_test = selected_data["y_test"]

y_pred = best_model.predict(X_test)
metrics = compute_metrics(y_test, y_pred)

summary_eval_df = pd.DataFrame(
    {
        "dataset": [best_dataset_name],
        "model": [best_model_name],
        "MAE": [metrics["MAE"]],
        "RMSE": [metrics["RMSE"]],
        "R2": [metrics["R2"]],
    }
)

display(summary_eval_df)
print(f"MAE  = {metrics['MAE']:.4f}")
print(f"RMSE = {metrics['RMSE']:.4f}")
print(f"R2   = {metrics['R2']:.4f}")

## 6. Phân tích phần dư và trực quan dự đoán

In [ ]:
residuals = y_test - y_pred
abs_error = np.abs(residuals)

diagnostics_df = pd.DataFrame(
    {
        "actual": y_test.values,
        "predicted": y_pred,
        "residual": residuals,
        "abs_error": abs_error,
    }
)

display(diagnostics_df.head())
display(diagnostics_df.describe().T)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.scatterplot(x=diagnostics_df["actual"], y=diagnostics_df["predicted"], ax=axes[0], alpha=0.6)
min_val = min(diagnostics_df["actual"].min(), diagnostics_df["predicted"].min())
max_val = max(diagnostics_df["actual"].max(), diagnostics_df["predicted"].max())
axes[0].plot([min_val, max_val], [min_val, max_val], "r--", linewidth=1.5)
axes[0].set_title("Thực tế và dự đoán")
axes[0].set_xlabel("Rings thực tế")
axes[0].set_ylabel("Rings dự đoán")

sns.histplot(diagnostics_df["residual"], kde=True, bins=30, ax=axes[1], color="#2a9d8f")
axes[1].axvline(0, color="red", linestyle="--", linewidth=1.2)
axes[1].set_title("Phân phối phần dư")
axes[1].set_xlabel("Phần dư")

sns.scatterplot(
    x=diagnostics_df["predicted"],
    y=diagnostics_df["residual"],
    ax=axes[2],
    alpha=0.6,
)
axes[2].axhline(0, color="red", linestyle="--", linewidth=1.2)
axes[2].set_title("Phần dư theo giá trị dự đoán")
axes[2].set_xlabel("Giá trị dự đoán")
axes[2].set_ylabel("Phần dư")

plt.tight_layout()
plt.show()

## 7. Phân tích sai số theo nhóm `Sex`

Mục tiêu: kiểm tra mô hình có sai lệch rõ ràng giữa các nhóm `F`, `I`, `M` hay không.

In [ ]:
sex_cols = [col for col in X_test.columns if col.startswith("Sex_")]

if len(sex_cols) > 0:
    sex_values = X_test[sex_cols].idxmax(axis=1).str.replace("Sex_", "", regex=False)

    group_df = pd.DataFrame(
        {
            "Sex": sex_values.values,
            "actual": y_test.values,
            "predicted": y_pred,
        }
    )
    group_df["abs_error"] = np.abs(group_df["actual"] - group_df["predicted"])

    group_metrics = (
        group_df.groupby("Sex")
        .agg(
            count=("actual", "size"),
            mae=("abs_error", "mean"),
            actual_mean=("actual", "mean"),
            pred_mean=("predicted", "mean"),
        )
        .reset_index()
        .sort_values("mae")
    )

    display(group_metrics)

    plt.figure(figsize=(8, 5))
    sns.barplot(data=group_metrics, x="Sex", y="mae", palette="Set2")
    plt.title("MAE theo nhóm giới tính")
    plt.xlabel("Nhóm giới tính")
    plt.ylabel("MAE")
    plt.tight_layout()
    plt.show()
else:
    group_metrics = pd.DataFrame()
    print("Không tìm thấy cột Sex_ trong tập đánh giá, bỏ qua phân tích theo nhóm giới tính.")

## 8. Đánh giá độ ổn định bằng bootstrap

Ước lượng khoảng tin cậy thực nghiệm cho `MAE` và `RMSE` trên tập test.

In [ ]:
rng = np.random.default_rng(42)
N_BOOTSTRAP = 500

mae_samples = []
rmse_samples = []

actual_vals = y_test.to_numpy()
pred_vals = np.asarray(y_pred)
n = len(actual_vals)

for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, n, size=n)
    y_b = actual_vals[idx]
    p_b = pred_vals[idx]

    mae_b = mean_absolute_error(y_b, p_b)
    rmse_b = np.sqrt(mean_squared_error(y_b, p_b))

    mae_samples.append(mae_b)
    rmse_samples.append(rmse_b)

mae_ci = np.percentile(mae_samples, [2.5, 97.5])
rmse_ci = np.percentile(rmse_samples, [2.5, 97.5])

print(f"Khoảng tin cậy 95% của MAE : [{mae_ci[0]:.4f}, {mae_ci[1]:.4f}]")
print(f"Khoảng tin cậy 95% của RMSE: [{rmse_ci[0]:.4f}, {rmse_ci[1]:.4f}]")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(mae_samples, bins=30, kde=True, ax=axes[0], color="#457b9d")
axes[0].set_title("Phân phối bootstrap của MAE")
axes[0].axvline(metrics["MAE"], color="red", linestyle="--", linewidth=1.2)

sns.histplot(rmse_samples, bins=30, kde=True, ax=axes[1], color="#e76f51")
axes[1].set_title("Phân phối bootstrap của RMSE")
axes[1].axvline(metrics["RMSE"], color="red", linestyle="--", linewidth=1.2)

plt.tight_layout()
plt.show()

## 9. Lưu báo cáo đánh giá

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

evaluation_report = {
    "timestamp": timestamp,
    "dataset": best_dataset_name,
    "model": best_model_name,
    "metrics": {
        "mae": float(metrics["MAE"]),
        "rmse": float(metrics["RMSE"]),
        "r2": float(metrics["R2"]),
    },
    "bootstrap_ci": {
        "mae_95": [float(mae_ci[0]), float(mae_ci[1])],
        "rmse_95": [float(rmse_ci[0]), float(rmse_ci[1])],
    },
}

if not group_metrics.empty:
    evaluation_report["group_metrics_by_sex"] = group_metrics.to_dict(orient="records")

report_path = OUTPUT_METRICS_PATH / "evaluation_report.json"
diagnostics_csv_path = OUTPUT_METRICS_PATH / "prediction_diagnostics.csv"
fig_path = OUTPUT_FIGURES_PATH / f"evaluation_residuals_{timestamp}.png"

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(evaluation_report, f, ensure_ascii=False, indent=2)

diagnostics_df.to_csv(diagnostics_csv_path, index=False)

plt.figure(figsize=(7, 6))
sns.scatterplot(data=diagnostics_df, x="actual", y="predicted", alpha=0.55)
plt.plot([min_val, max_val], [min_val, max_val], "r--", linewidth=1.2)
plt.title("Đánh giá: giá trị thực tế và dự đoán")
plt.xlabel("Rings thực tế")
plt.ylabel("Rings dự đoán")
plt.tight_layout()
plt.savefig(fig_path, dpi=160)
plt.show()

print("Đã lưu báo cáo đánh giá:")
print(f"- Tệp báo cáo: {report_path}")
print(f"- Tệp chẩn đoán: {diagnostics_csv_path}")
print(f"- Hình minh họa: {fig_path}")

## 10. Kết luận đánh giá

- Mô hình được kiểm tra bằng cả chỉ số trung bình (`MAE`, `RMSE`, `R2`) và phân phối sai số.
- Bootstrap cho thấy mức dao động kỳ vọng của `MAE` và `RMSE` trên mẫu dữ liệu tương tự.
- Phân tích theo nhóm `Sex` giúp nhận diện khả năng lệch hiệu năng giữa các phân khúc.
- Các tệp đánh giá đã được lưu trong thư mục `outputs/metrics` và `outputs/figures`.